# 🗂️ Notebook 5 — API Center Registration and Live Discovery Setup

This notebook registers the deployed specialist agents in **Azure API Center** as the governance catalog,  
and configures **live runtime discovery** for the orchestrator (no local snapshot file).

## Why this sequence matters
```
Specialists deployed (Notebook 3)
        │
        ▼ (register with metadata: capabilities, persona, trust, risk)
  Azure API Center  ←── Governance catalog / source of truth
        │
        ▼ (live discovery at runtime via MCP data-plane or REST)
  pf-orchestrator dynamically selects allowed agents
        │
        ▼
  Execute selected calls through governed gateway/runtime path
```

## What this notebook does
1. Verifies or creates API Center in the hub resource group
2. Registers each specialist as an API with governance metadata
3. Validates required metadata fields for orchestration decisions
4. Verifies live discovery by listing `pf-*` APIs from API Center
5. Persists live-discovery configuration for downstream orchestrator notebooks

In [ ]:
import sys, json, pathlib, hashlib, datetime, subprocess, re, tempfile, shutil

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing shared/utils.py and workshop/product-finder.")

repo_root = find_repo_root(pathlib.Path.cwd())
shared_dir = repo_root / "shared"
sys.path.insert(0, str(shared_dir))
import utils

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

def set_azd_env(key: str, value: str):
    p = subprocess.run(["azd", "env", "set", key, value], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(p.stderr or p.stdout).strip()}")

SUB_ID = azd_get("AZURE_SUBSCRIPTION_ID")
hub_rg = azd_get("AZURE_RESOURCE_GROUP")
account = azd_get("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
project = azd_get("SPOKE_AI_FOUNDRY_PROJECT_NAME")
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", f"https://{account}.services.ai.azure.com/api/projects/{project}")

apic_name_hint = azd_get_optional("APIC_NAME", "")
if not apic_name_hint:
    token = re.sub(r"[^a-z0-9]", "", hub_rg.lower())[:20] or "hub"
    apic_name_hint = f"apic-{token}"

deployed_raw = azd_get_optional("PF_DEPLOYED_SPECIALISTS", "[]")
try:
    deployed = json.loads(deployed_raw)
except Exception:
    deployed = []

utils.print_info(f"Repo root:          {repo_root}")
utils.print_info(f"Shared dir:         {shared_dir}")
utils.print_info(f"Hub RG:             {hub_rg}")
utils.print_info(f"API Center target:  {apic_name_hint}")
utils.print_info(f"Deployed specialists: {deployed}")

if not deployed:
    raise RuntimeError("No deployed specialists found in env. Run Notebook 3 first.")

### 1️⃣ Find API Center resource in hub resource group

In [ ]:
# We intentionally avoid relying on the optional 'az apic' CLI extension.
# Discovery uses ARM resource queries, and creation uses a Bicep deployment.
apic_extension_ready = False

apic_out = run(
    f"az resource list -g {hub_rg} "
    f"--resource-type Microsoft.ApiCenter/services -o json",
    "API Center query OK", "API Center query failed"
)

if not apic_out.success or not apic_out.json_data:
    utils.print_warning(
        "No API Center resource found in hub RG. Creating one now via Bicep deployment..."
    )

    run(
        "az provider register --namespace Microsoft.ApiCenter -o none",
        "Microsoft.ApiCenter provider registration started",
        "Microsoft.ApiCenter provider registration failed"
    )

    apic_template = (repo_root / "bicep" / "infra" / "modules" / "apic" / "apic.bicep").resolve()
    deploy_name = f"apic-bootstrap-{datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

    create_apic = run(
        f"az deployment group create -g {hub_rg} --name {deploy_name} "
        f"--template-file \"{apic_template}\" "
        f"--parameters apicServiceName={apic_name_hint} loadSampleMCPs=false -o json",
        f"API Center deployment succeeded: {apic_name_hint}",
        "API Center deployment failed"
    )
    if not create_apic.success:
        raise RuntimeError(
            "Unable to create API Center in hub RG via Bicep deployment. Confirm permissions and region support."
        )

    # Re-query after deployment to confirm the resource now exists.
    apic_verify = run(
        f"az resource list -g {hub_rg} "
        f"--resource-type Microsoft.ApiCenter/services -o json",
        "API Center verification query OK", "API Center verification query failed"
    )
    if not apic_verify.success or not apic_verify.json_data:
        raise RuntimeError("API Center deployment completed but the service was not discoverable in the hub RG.")

    apic_name = apic_verify.json_data[0]["name"]
    apic_found = True
    set_azd_env("APIC_NAME", apic_name)
    utils.print_ok(f"API Center ready: {apic_name}")
else:
    apic_name = apic_out.json_data[0]["name"]
    apic_found = True
    set_azd_env("APIC_NAME", apic_name)
    utils.print_ok(f"API Center found: {apic_name}")

### 2️⃣ Register deployed specialists in API Center

Each agent is registered as an API with governance metadata embedded in custom properties.  

### Metadata template used for specialist registration

The registration cell now works like a template + params contract:

- `governance-profile.template.json` = schema/defaults (template)
- `governance-profile.params.json` = per-agent values (params)

Both files are generated under:

- `workshop/product-finder/registry/governance-contract/`

Mandatory fields for each specialist profile:

- agent_name
- description
- capabilities
- supported_intents
- allowed_personas
- risk_tiers_supported
- orchestration_stage
- execution_order
- run_after
- requires_disclaimer
- trust_level
- verification_status
- enabled

If a deployed agent is missing required params, has unknown keys, or misses mandatory fields after merge, registration fails immediately.

```json
{
  "agent_name": "pf-<name>",
  "description": "Agent goal: ... Use when: ... Do not use when: ... Input expected: ... Output produced: ...",
  "capabilities": ["..."],
  "supported_intents": ["..."],
  "allowed_personas": ["external_customer", "internal_scientist"],
  "risk_tiers_supported": ["low", "elevated"],
  "orchestration_stage": "contextualization|retrieval|analysis|alignment|fulfillment",
  "execution_order": 100,
  "run_after": ["pf-other-agent"],
  "requires_disclaimer": false,
  "trust_level": "high",
  "verification_status": "verified",
  "enabled": true
}
```

In [ ]:
METADATA_TEMPLATE_VERSION = "3.1"

# Template (like Bicep): shared contract + defaults.
GOVERNANCE_PROFILE_TEMPLATE = {
    "agent_type": "specialist",
    "domain": "product-finder",
    "environment_allowlist": ["workshop", "dev", "prod"],
    "verification_status": "verified",
    "trust_level": "high",
    "supports_simulation": False,
    "auth_required": False,
    "requires_disclaimer": False,
    "enabled": True,
    "min_confidence_threshold": 0.75,
    "parallel_group": "sequential",
    "priority": 999,
    "execution_order": 500,
    "run_after": [],
    "depends_on": [],
    "required_context_fields": [],
    "produces": [],
    "preferred_data_sources": [],
}

# Required contract (like required params in Bicep).
MANDATORY_METADATA_FIELDS = {
    "agent_name",
    "description",
    "capabilities",
    "supported_intents",
    "allowed_personas",
    "risk_tiers_supported",
    "orchestration_stage",
    "execution_order",
    "run_after",
    "requires_disclaimer",
    "trust_level",
    "verification_status",
    "enabled",
}

# Allowed per-agent params keys; unknown keys fail fast so teams follow the template.
ALLOWED_AGENT_PARAM_KEYS = {
    "goal",
    "use_when",
    "do_not_use_when",
    "input_expected",
    "output_produced",
    "capabilities",
    "supported_intents",
    "conditional_intents",
    "required_context_fields",
    "produces",
    "preferred_data_sources",
    "allowed_personas",
    "risk_tiers_supported",
    "orchestration_stage",
    "execution_order",
    "priority",
    "parallel_group",
    "run_after",
    "fallback_agents",
    "auth_required",
    "supports_simulation",
    "requires_disclaimer",
    "min_confidence_threshold",
    "verification_status",
    "trust_level",
    "enabled",
}


def build_description(goal: str, use_when: str, do_not_use_when: str, input_expected: str, output_produced: str) -> str:
    return (
        f"Agent goal: {goal} "
        f"Use when: {use_when} "
        f"Do not use when: {do_not_use_when} "
        f"Input expected: {input_expected} "
        f"Output produced: {output_produced}"
    )


# Params (like Bicep params): teams fill these values per specialist.
AGENT_METADATA_PARAMS = {
    "pf-contextualizer": {
        "goal": "Turn raw user asks into a complete, machine-actionable request context for downstream specialists.",
        "use_when": "Always as the first specialist step to classify intent, extract entities, and detect missing context or elevated risk.",
        "do_not_use_when": "Intent, entities, and risk are already complete and validated for this turn.",
        "input_expected": "Raw user request plus governance context (persona and disclaimer_accepted), optionally prior turns.",
        "output_produced": "Structured context object with intent, entities, missing_context, risk_tier, contextualized_query, and confidence.",
        "capabilities": ["intent_extraction", "entity_extraction", "query_completion", "risk_classification"],
        "supported_intents": ["recommendation", "compatibility", "sample_request", "out_of_domain"],
        "required_context_fields": ["query_text"],
        "produces": ["intent", "entities", "missing_context", "risk_tier", "contextualized_query", "confidence"],
        "preferred_data_sources": ["chat_history", "rag_context"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "risk_tiers_supported": ["low", "elevated"],
        "orchestration_stage": "contextualization",
        "execution_order": 100,
        "priority": 10,
        "run_after": [],
        "fallback_agents": ["pf-aligner"],
    },
    "pf-product-intelligence": {
        "goal": "Provide evidence-based product info, recommendations, and behavioral analysis.",
        "use_when": "Product information, recommendations, compatibility support, and sample-request product validation.",
        "do_not_use_when": "Quick yes/no verdict without detailed analysis; use pf-compatibility for simple checks.",
        "input_expected": "Contextualized query, product entities, persona, and usage conditions.",
        "output_produced": "Top candidates, recommendation summary, evidence, behavior analysis, and confidence.",
        "capabilities": ["product_recommendation", "product_search", "catalog_retrieval", "product_behavior_analysis", "formulation_compatibility", "product_chemistry_analysis"],
        "supported_intents": ["recommendation", "product_information", "compatibility","sample_request"],
        "conditional_intents": {"compatibility": {"personas": ["internal_scientist"], "requires_context": ["formulation_analysis", "product_behavior"]}},
        "required_context_fields": ["contextualized_query", "entities"],
        "produces": ["top_products", "recommendation_summary", "evidence_refs", "confidence", "product_behavior_analysis", "ingredient_compatibility_notes"],
        "preferred_data_sources": ["product_datasheets", "functional_descriptions", "technical_descriptions", "ingredient_lists", "formulation_notes"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "risk_tiers_supported": ["low", "elevated"],
        "orchestration_stage": "retrieval",
        "execution_order": 300,
        "priority": 20,
        "parallel_group": "recommendation-core",
        "run_after": ["pf-contextualizer"],
        "fallback_agents": ["pf-contextualizer"],
    },
    "pf-compatibility": {
        "goal": "Assess whether product combinations are safe and compatible under risk-aware governance.",
        "use_when": "Intent is compatibility or mixing safety and elevated-risk reasoning is needed for two or more products.",
        "do_not_use_when": "User only requests a basic recommendation, general catalog facts, or non-risk transaction handling.",
        "input_expected": "Contextualized compatibility question with identified product entities, plus validated product context from pf-product-intelligence when available.",
        "output_produced": "Compatibility verdict, safety_rationale, warning, requires_vet_guidance, and confidence with risk-aware guidance.",
        "capabilities": ["compatibility_check", "risk_assessment", "confidence_scoring"],
        "supported_intents": ["compatibility"],
        "required_context_fields": ["contextualized_query", "entities.products"],
        "produces": ["verdict", "safety_rationale", "warning", "requires_vet_guidance", "confidence"],
        "preferred_data_sources": ["entity_graph", "compatibility_knowledge", "upstream_product_validation"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "risk_tiers_supported": ["elevated"],
        "requires_disclaimer": True,
        "min_confidence_threshold": 0.90,
        "orchestration_stage": "analysis",
        "execution_order": 450,
        "priority": 30,
        "parallel_group": "risk-core",
        "run_after": ["pf-contextualizer", "pf-product-intelligence"],
        "fallback_agents": ["pf-product-intelligence"],
    },
    "pf-sample-request": {
        "goal": "Handle authenticated product sample request fulfillment outcomes for external customers.",
        "use_when": "Intent is sample_request and the user asks to initiate product sample fulfillment.",
        "do_not_use_when": "Request is recommendation-only, compatibility-only, or the persona is not an authenticated external customer.",
        "input_expected": "Validated product details from pf-product-intelligence, persona, and authentication context required for sample workflow.",
        "output_produced": "request_accepted flag, product confirmation, confirmation_number, estimated_delivery, and auth_status.",
        "capabilities": ["sample_request_processing", "authentication_verification"],
        "supported_intents": ["sample_request"],
        "required_context_fields": ["validated_product.product_id", "validated_product.product_name", "persona", "auth_context"],
        "produces": ["request_accepted", "confirmation_number", "estimated_delivery", "auth_status"],
        "preferred_data_sources": ["upstream_product_validation"],
        "allowed_personas": ["external_customer"],
        "risk_tiers_supported": ["low"],
        "orchestration_stage": "fulfillment",
        "execution_order": 900,
        "priority": 40,
        "run_after": ["pf-contextualizer", "pf-product-intelligence"],
        "fallback_agents": ["pf-product-intelligence"],
        "auth_required": True,
    },
    "pf-aligner": {
        "goal": "Produce a final, intent-aligned, governance-safe response bundle for the end user.",
        "use_when": "Always as the final specialist step to consolidate specialist outputs and ensure final response matches original intent.",
        "do_not_use_when": "The conversation has unresolved missing_context and no downstream specialist outputs should be finalized yet.",
        "input_expected": "Original user query, contextualized intent, and upstream specialist outputs including governance notices.",
        "output_produced": "Final answer bundle with intent_matched, governance_notices, removed_claims, and confidence.",
        "capabilities": ["intent_alignment", "response_validation", "formatting"],
        "supported_intents": ["recommendation", "compatibility", "sample_request", "product_information", "out_of_domain"],
        "required_context_fields": ["original_query", "draft_response_bundle"],
        "produces": ["final_answer", "intent_matched", "governance_notices", "removed_claims", "confidence"],
        "preferred_data_sources": ["agent_outputs"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "risk_tiers_supported": ["low", "elevated"],
        "orchestration_stage": "alignment",
        "execution_order": 800,
        "priority": 90,
        "run_after": ["pf-product-intelligence", "pf-compatibility", "pf-sample-request"],
        "fallback_agents": ["pf-contextualizer"],
    },
}

# Persist template + params artifacts so teams can treat them like infra template files.
registry_contract_dir = (repo_root / "workshop" / "product-finder" / "registry" / "governance-contract").resolve()
registry_contract_dir.mkdir(parents=True, exist_ok=True)

(template_path := registry_contract_dir / "governance-profile.template.json").write_text(
    json.dumps(
        {
            "metadataSchemaVersion": METADATA_TEMPLATE_VERSION,
            "mandatoryFields": sorted(MANDATORY_METADATA_FIELDS),
            "allowedAgentParamKeys": sorted(ALLOWED_AGENT_PARAM_KEYS),
            "templateDefaults": GOVERNANCE_PROFILE_TEMPLATE,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

(params_path := registry_contract_dir / "governance-profile.params.json").write_text(
    json.dumps(
        {
            "metadataSchemaVersion": METADATA_TEMPLATE_VERSION,
            "agents": AGENT_METADATA_PARAMS,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

utils.print_info(f"Governance template written: {template_path}")
utils.print_info(f"Governance params written:   {params_path}")


def build_profile_from_template(agent_name: str, params: dict) -> dict:
    unknown_keys = sorted(set(params.keys()) - ALLOWED_AGENT_PARAM_KEYS)
    if unknown_keys:
        raise RuntimeError(f"{agent_name}: unknown params keys not allowed by template: {unknown_keys}")

    required_description_params = [
        "goal",
        "use_when",
        "do_not_use_when",
        "input_expected",
        "output_produced",
    ]
    missing_description_params = [k for k in required_description_params if not str(params.get(k, "")).strip()]
    if missing_description_params:
        raise RuntimeError(f"{agent_name}: missing description params: {missing_description_params}")

    profile = {**GOVERNANCE_PROFILE_TEMPLATE}
    profile.update({k: v for k, v in params.items() if k in ALLOWED_AGENT_PARAM_KEYS})
    profile["agent_name"] = agent_name
    profile["description"] = build_description(
        goal=str(params["goal"]),
        use_when=str(params["use_when"]),
        do_not_use_when=str(params["do_not_use_when"]),
        input_expected=str(params["input_expected"]),
        output_produced=str(params["output_produced"]),
    )
    profile["depends_on"] = list(profile.get("run_after") or [])

    missing_fields = [f for f in sorted(MANDATORY_METADATA_FIELDS) if f not in profile]
    if missing_fields:
        raise RuntimeError(f"{agent_name}: profile missing mandatory fields after template merge: {missing_fields}")

    return profile


AGENT_GOVERNANCE_METADATA = {}
for name, params in AGENT_METADATA_PARAMS.items():
    AGENT_GOVERNANCE_METADATA[name] = build_profile_from_template(name, params)

if not apic_found:
    raise RuntimeError("API Center was not initialized. Run Cell 4 first.")

registered_entries = []
api_tmp_dir = pathlib.Path(tempfile.mkdtemp(prefix="pf-apic-")).resolve()
utils.print_info(f"Using transient API payload dir: {api_tmp_dir}")

for agent_name in deployed:
    meta = AGENT_GOVERNANCE_METADATA.get(agent_name, {})

    if not meta:
        raise RuntimeError(
            f"No governance metadata params found for deployed agent '{agent_name}'. "
            "Add this agent to AGENT_METADATA_PARAMS to satisfy the template contract."
        )

    # Register in API Center via ARM REST.
    api_id = agent_name.replace("-", "")
    api_url = (
        f"https://management.azure.com/subscriptions/{SUB_ID}"
        f"/resourceGroups/{hub_rg}/providers/Microsoft.ApiCenter/services/{apic_name}"
        f"/workspaces/default/apis/{api_id}?api-version=2024-06-01-preview"
    )
    api_body = {
        "properties": {
            "title": agent_name,
            "kind": "rest",
            "summary": f"{agent_name} specialist API",
            "description": meta.get("description", f"Product Finder specialist agent registration for {agent_name}"),
            "customProperties": {
                "domain": meta.get("domain", "product-finder"),
                "agentName": agent_name,
                "description": meta.get("description", ""),
                "verificationStatus": meta.get("verification_status", "unknown"),
                "trustLevel": meta.get("trust_level", "unknown"),
                "orchestrationStage": meta.get("orchestration_stage", "unknown"),
                "executionOrder": str(meta.get("execution_order", 500)),
                "priority": str(meta.get("priority", 999)),
                "supportedIntents": ",".join(meta.get("supported_intents", [])),
                "allowedPersonas": ",".join(meta.get("allowed_personas", [])),
                "riskTiersSupported": ",".join(meta.get("risk_tiers_supported", [])),
                "runAfter": ",".join(meta.get("run_after", [])),
                "requiresDisclaimer": str(meta.get("requires_disclaimer", False)).lower(),
                "authRequired": str(meta.get("auth_required", False)).lower(),
                "supportsSimulation": str(meta.get("supports_simulation", False)).lower(),
                "enabled": str(meta.get("enabled", True)).lower(),
                "foundryEndpoint": FOUNDRY_EP,
                "metadataSchemaVersion": METADATA_TEMPLATE_VERSION,
                "governanceProfile": json.dumps(meta, separators=(",", ":"), sort_keys=True),
            },
        }
    }

    api_body_path = api_tmp_dir / f"{api_id}.json"
    api_body_path.write_text(json.dumps(api_body), encoding="utf-8")

    create_api = run(
        f"az rest --method put --url \"{api_url}\" --body \"@{api_body_path}\" -o json",
        f"  Registered {agent_name} in API Center",
        f"  API Center registration failed for {agent_name}"
    )
    utils.print_info(f"  API Center: {agent_name} -> {'OK' if create_api.success else 'failed'}")

    # Build registry entry regardless (used for runtime snapshot)
    entry = {
        "id": agent_name,
        "agent_name": agent_name,
        "display_name": agent_name.replace("-", " ").title(),
        "status": "active",
        "version": "1.0.0",
        "metadata_schema_version": METADATA_TEMPLATE_VERSION,
        "foundry_endpoint": FOUNDRY_EP,
        **meta,
    }
    registered_entries.append(entry)
    utils.print_ok(f"  {agent_name}: registry entry built")

utils.print_info(f"\nTotal registered: {len(registered_entries)} agents")
shutil.rmtree(api_tmp_dir, ignore_errors=True)
utils.print_info("Cleaned transient API payload dir")

In [ ]:
# ── 3️⃣  Validate metadata and configure live discovery (no snapshot) ────────
REQUIRED_FIELDS = {
    "id",
    "agent_name",
    "status",
    "description",
    "domain",
    "capabilities",
    "supported_intents",
    "allowed_personas",
    "verification_status",
    "trust_level",
    "risk_tiers_supported",
    "requires_disclaimer",
    "min_confidence_threshold",
    "orchestration_stage",
    "execution_order",
    "run_after",
    "priority",
    "auth_required",
    "supports_simulation",
    "enabled",
}

errors = []
for entry in registered_entries:
    missing = REQUIRED_FIELDS - set(entry.keys())
    if missing:
        errors.append(f"{entry['id']}: missing fields {sorted(missing)}")
if errors:
    raise RuntimeError("Registry schema validation FAILED:\n" + "\n".join(errors))

# Enforce deterministic order for orchestration planning.
registered_entries = sorted(
    registered_entries,
    key=lambda x: (int(x.get("execution_order", 500)), int(x.get("priority", 999))),
)
utils.print_ok("Registry schema validation passed")

# Live discovery check from API Center: list all APIs and keep pf-*.
apis_url = (
    f"https://management.azure.com/subscriptions/{SUB_ID}"
    f"/resourceGroups/{hub_rg}/providers/Microsoft.ApiCenter/services/{apic_name}"
    f"/workspaces/default/apis?api-version=2024-06-01-preview"
)

apis_out = run(
    f"az rest --method get --url \"{apis_url}\" -o json",
    "API Center live discovery query OK",
    "API Center live discovery query failed"
)
if not apis_out.success:
    raise RuntimeError("Failed to query API Center live discovery endpoint.")

api_items = []
if isinstance(apis_out.json_data, dict):
    api_items = apis_out.json_data.get("value", [])
elif isinstance(apis_out.json_data, list):
    api_items = apis_out.json_data

pf_items = []
for item in api_items:
    name = (item.get("name") or "").lower()
    title = ((item.get("properties") or {}).get("title") or "").lower()
    if name.startswith("pf") or title.startswith("pf-"):
        pf_items.append(item)

utils.print_info(f"Live-discovered APIs total: {len(api_items)}")
utils.print_info(f"Live-discovered pf-* APIs: {len(pf_items)}")
if len(pf_items) < len(deployed):
    utils.print_warning("Live discovery returned fewer pf-* APIs than deployed specialists. Check registration consistency.")

print("\n── LIVE DISCOVERY: PF AGENTS IN API CENTER ─────────────────────────")
for item in pf_items:
    props = item.get("properties") or {}
    custom = props.get("customProperties") or {}
    print(f"  ✅ {props.get('title', item.get('name', '<unknown>'))}")
    print(
        f"      stage={custom.get('orchestrationStage','?')} "
        f"order={custom.get('executionOrder','?')} "
        f"priority={custom.get('priority','?')} "
        f"trust={custom.get('trustLevel','?')}"
    )

# Persist only runtime env settings; no local snapshot file is required.
for k, v in {
    "PF_DISCOVERY_MODE": "api_center_live",
    "PF_API_CENTER_NAME": apic_name,
    "PF_API_CENTER_WORKSPACE": "default",
    "PF_API_CENTER_API_VERSION": "2024-06-01-preview",
    "PF_API_CENTER_APIS_URL": apis_url,
}.items():
    p = subprocess.run(["azd", "env", "set", k, str(v)], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist {k} to azd env: {(p.stderr or p.stdout).strip()}")
utils.print_ok("Persisted live-discovery runtime settings to azd env")

print()
utils.print_ok("✅ API Center registration and live discovery setup COMPLETE. Proceed to Notebook 6.")